# 🏆 FIFA World Cup 2026 — Winner Predictor

**Model:** ELO Rating System + Monte Carlo Simulation  
**Data:** 900+ FIFA World Cup matches (1930–2022)  
**Format:** 48-team, 12-group expanded tournament

---

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import json
import math
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from collections import defaultdict

# Pretty plot style
plt.rcParams.update({
    'figure.facecolor': '#0e1520',
    'axes.facecolor':   '#111927',
    'axes.edgecolor':   '#1e2d42',
    'axes.labelcolor':  '#8b9cbf',
    'xtick.color':      '#8b9cbf',
    'ytick.color':      '#8b9cbf',
    'text.color':       '#f0f4ff',
    'grid.color':       '#1e2d42',
    'grid.linewidth':   0.6,
    'font.family':      'sans-serif',
    'figure.dpi':       120,
})

print('Setup complete.')

## 2. Load Historical Data

In [ ]:
from data.historical_matches import get_matches, WC_WINNERS, get_all_teams

matches = get_matches()
print(f'Total matches loaded: {len(matches)}')
print(f'Tournaments covered: {sorted(set(m[0] for m in matches))}')
print(f'\nWC Winners by year:')
for yr, winner in sorted(WC_WINNERS.items()):
    print(f'  {yr}: {winner}')

In [ ]:
# Count matches per team (appearances)
team_appearances = defaultdict(int)
for _, _, t1, t2, _, _ in matches:
    team_appearances[t1] += 1
    team_appearances[t2] += 1

# Plot top 20 by appearances
top_apps = sorted(team_appearances.items(), key=lambda x: x[1], reverse=True)[:20]
teams_a, counts_a = zip(*top_apps)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#f5c842' if t in ['Brazil','Germany','Argentina','France','Italy'] else '#4fa3f7'
          for t in teams_a]
bars = ax.barh(teams_a, counts_a, color=colors, alpha=0.85, edgecolor='none')
ax.set_xlabel('Total WC Matches Played')
ax.set_title('Most Experienced World Cup Nations (1930-2022)', pad=12, fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.4)
for bar, val in zip(bars, counts_a):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9, color='#8b9cbf')
plt.tight_layout()
plt.savefig('output/appearances.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Compute ELO Ratings

In [ ]:
from model.elo import get_final_ratings, compute_elo_ratings

ratings = get_final_ratings()

# Show top 20
top_elo = sorted(ratings.items(), key=lambda x: x[1], reverse=True)[:20]

fig, ax = plt.subplots(figsize=(12, 6))
teams_e, elos_e = zip(*top_elo)
bar_colors = ['#f5c842' if i < 3 else '#4fa3f7' if i < 8 else '#34d399'
              for i in range(len(teams_e))]
bars = ax.bar(range(len(teams_e)), elos_e, color=bar_colors, alpha=0.85)
ax.set_xticks(range(len(teams_e)))
ax.set_xticklabels(teams_e, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('ELO Rating')
ax.set_title('Top 20 Teams by ELO Rating (WC History + Recent Form)', fontsize=14, fontweight='bold')
ax.axhline(1500, color='#f87171', linestyle='--', alpha=0.5, linewidth=1, label='Baseline ELO (1500)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
for bar, val in zip(bars, elos_e):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=8, color='#8b9cbf')
plt.tight_layout()
plt.savefig('output/elo_ratings.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nTop 5 ELO ratings:')
for team, elo in top_elo[:5]:
    print(f'  {team}: {elo:.1f}')

## 4. ELO History — How Ratings Evolved Over Tournaments

In [ ]:
from model.elo import BASE_ELO, update_elo

key_teams = ['Brazil', 'Germany', 'Argentina', 'France', 'Italy',
             'Spain', 'England', 'Netherlands']

raw_ratings = defaultdict(lambda: BASE_ELO)
history = {t: [] for t in key_teams}
current_years = {}

for year, stage, team1, team2, s1, s2 in matches:
    r1 = raw_ratings[team1]
    r2 = raw_ratings[team2]
    new_r1, new_r2 = update_elo(r1, r2, s1, s2, stage, year)
    raw_ratings[team1] = new_r1
    raw_ratings[team2] = new_r2
    
    if year not in current_years:
        current_years[year] = {}
    if team1 in key_teams:
        current_years[year][team1] = new_r1
    if team2 in key_teams:
        current_years[year][team2] = new_r2

all_years = sorted(current_years.keys())
running = {t: BASE_ELO for t in key_teams}

timeline = {t: [] for t in key_teams}
for yr in all_years:
    yr_data = current_years.get(yr, {})
    for t in key_teams:
        if t in yr_data:
            running[t] = yr_data[t]
        timeline[t].append(running[t])

colors_hist = ['#f5c842','#4fa3f7','#34d399','#f472b6','#a78bfa','#fb923c','#f87171','#38bdf8']

fig, ax = plt.subplots(figsize=(14, 7))
for i, team in enumerate(key_teams):
    ax.plot(all_years, timeline[team], marker='o', markersize=4,
            linewidth=2, label=team, color=colors_hist[i], alpha=0.9)

ax.axhline(1500, color='#4a5568', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xlabel('World Cup Year')
ax.set_ylabel('ELO Rating')
ax.set_title('ELO Rating Evolution — Top Nations (1930–2022)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10, framealpha=0.3)
ax.grid(True, alpha=0.3)
ax.set_xticks(all_years)
ax.set_xticklabels(all_years, rotation=45, ha='right')
plt.tight_layout()
plt.savefig('output/elo_history.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Match Probability Analysis

In [ ]:
from model.predictor import match_probabilities, head_to_head_analysis

# Head-to-head analysis for interesting matchups
matchups = [
    ('Brazil',      'Argentina'),
    ('France',      'England'),
    ('Germany',     'Spain'),
    ('Netherlands', 'Belgium'),
    ('Portugal',    'France'),
    ('Argentina',   'Germany'),
    ('Morocco',     'France'),
    ('Japan',       'Germany'),
]

print(f'\n{"Matchup":<35} {"Win %":>8} {"Draw %":>8} {"Loss %":>8}')
print('-' * 65)
h2h_results = []
for a, b in matchups:
    r = head_to_head_analysis(a, b, ratings)
    h2h_results.append(r)
    print(f'{a} vs {b:<22} {r["p_win_a"]:>8.1f}% {r["p_draw"]:>7.1f}% {r["p_win_b"]:>7.1f}%')

In [ ]:
# Visualize H2H probabilities
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for ax, r in zip(axes, h2h_results):
    vals = [r['p_win_a'], r['p_draw'], r['p_win_b']]
    colors_h2h = ['#4fa3f7', '#a78bfa', '#f87171']
    labels_h2h = [r['team_a'], 'Draw', r['team_b']]
    wedges, texts = ax.pie(
        vals, labels=None,
        colors=colors_h2h, startangle=90,
        wedgeprops={'edgecolor': '#0e1520', 'linewidth': 2},
    )
    ax.set_title(f"{r['team_a']}\nvs {r['team_b']}", fontsize=8, pad=4)
    # Annotate center
    ax.text(0, 0, f"{vals[0]:.0f}%", ha='center', va='center', fontsize=12,
            fontweight='bold', color='#4fa3f7')

# Legend
legend_patches = [
    mpatches.Patch(color='#4fa3f7', label='Team A Win'),
    mpatches.Patch(color='#a78bfa', label='Draw'),
    mpatches.Patch(color='#f87171', label='Team B Win'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           framealpha=0.3, bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Head-to-Head Match Probabilities', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/h2h_probs.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Run Monte Carlo Simulation

In [ ]:
# Load pre-computed predictions (or run fresh with fewer sims for demo)
import json

try:
    with open('output/predictions.json', encoding='utf-8') as f:
        data = json.load(f)
    predictions = list(data['predictions'].values())
    n_sims = data['meta']['n_simulations']
    print(f'Loaded pre-computed predictions ({n_sims:,} simulations)')
except FileNotFoundError:
    print('predictions.json not found. Running 5,000 sims (quick demo)...')
    from model.simulator import run_monte_carlo
    preds_dict = run_monte_carlo(ratings, n_sims=5_000)
    predictions = list(preds_dict.values())
    n_sims = 5_000

# Sort by champion %
predictions.sort(key=lambda x: x['champion_pct'], reverse=True)
print(f'\nTop 10 Championship Predictions:')
print(f'{"Rank":<5} {"Team":<25} {"ELO":>6} {"Champion%":>10} {"Final%":>8} {"SF%":>6}')
print('-' * 62)
for i, t in enumerate(predictions[:10], 1):
    print(f'{i:<5} {t["team"]:<25} {t["elo"]:>6.0f} {t["champion_pct"]:>10.2f}% '
          f'{t["finalist_pct"]:>7.2f}% {t["semi_pct"]:>5.2f}%')

## 7. Championship Probability Chart

In [ ]:
top20 = predictions[:20]
teams_p = [t['team'] for t in top20]
champ_p = [t['champion_pct'] for t in top20]
final_p = [t['finalist_pct'] for t in top20]
semi_p  = [t['semi_pct'] for t in top20]

x = np.arange(len(teams_p))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 7))
b1 = ax.bar(x - width, champ_p, width, label='Champion %', color='#f5c842', alpha=0.9)
b2 = ax.bar(x,          final_p, width, label='Finalist %', color='#4fa3f7', alpha=0.9)
b3 = ax.bar(x + width,  semi_p,  width, label='Semi-Final %', color='#34d399', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(teams_p, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Probability (%)')
ax.set_title(f'FIFA WC 2026 Stage Probabilities — Top 20 Teams ({n_sims:,} simulations)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.3)
ax.grid(axis='y', alpha=0.4)

# Annotate champion bars
for bar, val in zip(b1, champ_p):
    if val > 2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=7.5, color='#f5c842')

plt.tight_layout()
plt.savefig('output/championship_probs.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Group Analysis

In [ ]:
groups = data['groups']
team_map = {t['team']: t for t in predictions}

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()

group_colors = {
    'A':'#4fa3f7','B':'#34d399','C':'#f5c842','D':'#f472b6',
    'E':'#a78bfa','F':'#fb923c','G':'#f87171','H':'#38bdf8',
    'I':'#4ade80','J':'#fbbf24','K':'#c084fc','L':'#f9a8d4',
}

for ax, (grp, grp_teams) in zip(axes, groups.items()):
    color = group_colors.get(grp, '#4fa3f7')
    sorted_grp = sorted(grp_teams, key=lambda t: team_map.get(t, {}).get('champion_pct', 0), reverse=True)
    pcts = [team_map.get(t, {}).get('champion_pct', 0) for t in sorted_grp]
    bars = ax.barh(sorted_grp, pcts, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(f'Group {grp}', fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('Champion %', fontsize=9)
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)
    for bar, val in zip(bars, pcts):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}%', va='center', fontsize=8.5, color='#8b9cbf')

fig.suptitle('Championship Probability by Group — FIFA WC 2026', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/group_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. ELO vs Championship Probability Scatter

In [ ]:
all_teams_list = predictions
elos    = [t['elo'] for t in all_teams_list]
champs  = [t['champion_pct'] for t in all_teams_list]
groups_list = [t['group'] for t in all_teams_list]
names   = [t['team'] for t in all_teams_list]

unique_grps = sorted(set(groups_list))
cmap = plt.cm.get_cmap('tab20', len(unique_grps))
grp_idx = {g: i for i, g in enumerate(unique_grps)}
colors_s = [cmap(grp_idx[g]) for g in groups_list]

fig, ax = plt.subplots(figsize=(13, 8))
sc = ax.scatter(elos, champs, c=colors_s, s=80, alpha=0.85, edgecolors='white', linewidths=0.5)

# Label top 12
for t in all_teams_list[:12]:
    ax.annotate(t['team'], (t['elo'], t['champion_pct']),
                textcoords='offset points', xytext=(6, 4),
                fontsize=8, color='#f0f4ff', alpha=0.9)

ax.set_xlabel('ELO Rating', fontsize=12)
ax.set_ylabel('Championship Probability (%)', fontsize=12)
ax.set_title('ELO Rating vs Championship Probability — All 48 Teams', fontsize=13, fontweight='bold')

legend_patches = [mpatches.Patch(color=cmap(grp_idx[g]), label=f'Group {g}') for g in unique_grps]
ax.legend(handles=legend_patches, loc='upper left', ncol=2, fontsize=8.5, framealpha=0.3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/elo_vs_champion.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Upset Analysis — Lower ELO Teams' Chances

In [ ]:
# Rank by ELO then see how champion% compares
by_elo = sorted(all_teams_list, key=lambda t: t['elo'], reverse=True)
elo_ranks  = list(range(1, len(by_elo) + 1))
champ_vals = [t['champion_pct'] for t in by_elo]
team_names_elo = [t['team'] for t in by_elo]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: ELO rank vs champion%
ax1.scatter(elo_ranks, champ_vals, c='#4fa3f7', s=60, alpha=0.7)
for i, (rank, val, name) in enumerate(zip(elo_ranks, champ_vals, team_names_elo)):
    if val > 5 or rank < 6:
        ax1.annotate(name, (rank, val), textcoords='offset points',
                     xytext=(4, 3), fontsize=7.5, color='#f0f4ff')
ax1.set_xlabel('ELO Rank (1 = highest)')
ax1.set_ylabel('Championship %')
ax1.set_title('ELO Rank vs Championship Probability')
ax1.grid(True, alpha=0.3)

# Right: "Upset potential" — teams ranked >15 by ELO but with >1% champ chance
upset_teams = [(t['team'], t['champion_pct'], t['elo'])
               for i, t in enumerate(by_elo) if i >= 12 and t['champion_pct'] > 0.5]
upset_teams.sort(key=lambda x: x[1], reverse=True)

if upset_teams:
    ut_names = [u[0] for u in upset_teams]
    ut_pcts  = [u[1] for u in upset_teams]
    bars2 = ax2.barh(ut_names, ut_pcts, color='#f472b6', alpha=0.85)
    ax2.set_xlabel('Championship %')
    ax2.set_title('Potential Dark Horses\n(Outside Top 12 by ELO, > 0.5% champion %)')
    ax2.invert_yaxis()
    ax2.grid(axis='x', alpha=0.3)
    for bar, val in zip(bars2, ut_pcts):
        ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}%', va='center', fontsize=9, color='#f472b6')
else:
    ax2.text(0.5, 0.5, 'No strong dark horses found', ha='center', va='center',
             transform=ax2.transAxes, fontsize=11, color='#8b9cbf')
    ax2.set_title('Dark Horses')

plt.suptitle('Upset Potential & Dark Horses — WC 2026', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/upset_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Summary Dashboard

In [ ]:
top15 = predictions[:15]

fig = plt.figure(figsize=(18, 10))
gs  = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

# Panel 1: Championship probabilities (top 15)
ax1 = fig.add_subplot(gs[0, :])
teams15  = [t['team'] for t in top15]
champ15  = [t['champion_pct'] for t in top15]
final15  = [t['finalist_pct'] for t in top15]
semi15   = [t['semi_pct'] for t in top15]

x15 = np.arange(len(teams15))
w = 0.25
ax1.bar(x15 - w,  champ15, w, label='Champion %',  color='#f5c842', alpha=0.9)
ax1.bar(x15,      final15, w, label='Finalist %',  color='#4fa3f7', alpha=0.9)
ax1.bar(x15 + w,  semi15,  w, label='Semi-Final %', color='#34d399', alpha=0.9)
ax1.set_xticks(x15)
ax1.set_xticklabels(teams15, rotation=30, ha='right', fontsize=10)
ax1.set_title(f'Top 15 Championship Favourites — {n_sims:,} Monte Carlo Simulations', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9, framealpha=0.3)
ax1.grid(axis='y', alpha=0.4)

# Panel 2: ELO ratings (top 15)
ax2 = fig.add_subplot(gs[1, 0])
elos15 = [t['elo'] for t in top15]
bar_c = ['#f5c842','#f5c842','#f5c842'] + ['#4fa3f7'] * 12
ax2.barh(teams15[::-1], elos15[::-1], color=bar_c[::-1], alpha=0.85)
ax2.set_xlabel('ELO Rating')
ax2.set_title('ELO Ratings', fontsize=11, fontweight='bold')
ax2.axvline(1500, color='#f87171', linestyle='--', alpha=0.5, linewidth=1)
ax2.grid(axis='x', alpha=0.4)

# Panel 3: Cumulative probability pie (top 8 + rest)
ax3 = fig.add_subplot(gs[1, 1])
pie_teams = [t['team'] for t in predictions[:8]] + ['Rest of World']
pie_vals  = [t['champion_pct'] for t in predictions[:8]]
pie_vals.append(sum(t['champion_pct'] for t in predictions[8:]))
pie_colors = ['#f5c842','#4fa3f7','#34d399','#f472b6','#a78bfa','#fb923c','#f87171','#38bdf8','#4a5568']
ax3.pie(pie_vals, labels=None, colors=pie_colors, startangle=90,
        wedgeprops={'edgecolor':'#0e1520','linewidth':2},
        autopct=lambda p: f'{p:.1f}%' if p > 3 else '')
ax3.set_title('Championship Share', fontsize=11, fontweight='bold')
legend3 = [mpatches.Patch(color=pie_colors[i], label=pie_teams[i]) for i in range(len(pie_teams))]
ax3.legend(handles=legend3, loc='lower center', bbox_to_anchor=(0.5, -0.35),
           fontsize=8, ncol=2, framealpha=0.3)

fig.suptitle('FIFA World Cup 2026 — Winner Prediction Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('output/summary_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n All plots saved to output/ directory.')
print('\n Prediction summary:')
print(f'  Favourite:     {predictions[0]["team"]} ({predictions[0]["champion_pct"]:.2f}%)')
print(f'  2nd favourite: {predictions[1]["team"]} ({predictions[1]["champion_pct"]:.2f}%)')
print(f'  3rd favourite: {predictions[2]["team"]} ({predictions[2]["champion_pct"]:.2f}%)')
total = sum(t['champion_pct'] for t in predictions)
print(f'  Probabilities sum to: {total:.1f}%')

---
## Model Notes

| Parameter | Value |
|-----------|-------|
| Simulations | 50,000 |
| Historical data | 1930–2022 (900+ matches) |
| ELO K-factor (Final) | 60 |
| ELO K-factor (Group) | 30 |
| Recency weight (post-2014) | 2× |
| Draw probability model | Dixon-Coles correction |
| Goal model | Poisson sampling |
| Penalty shootouts | 52/48 to higher ELO |

> ⚠️ **Disclaimer:** These are statistical estimates based on historical WC performance only. Actual player fitness, injuries, tactical matchups, and luck are not modelled.
